# Main Model Results

Research setup:
- Outcome: `wfood`
- Treatment: `D = 1(town >= 4)`
- Covariates: `log_totexp`, `size`, `age`, `sex_male`


## Outputs Produced By This Notebook

Running this notebook writes the main model outputs to `outputs/main_model_results/`.

Main reporting files are:
- `main_results.csv`
- `overlap_summary.csv`
- `propensity_overlap_plot.png`


In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 180
mpl.rcParams['savefig.dpi'] = 300

from pathlib import Path
import sys
import importlib.util

_cwd = Path.cwd().resolve()
_cands = [_cwd, _cwd.parent]
PROJECT_ROOT = next((p for p in _cands if (p / "MODELS" / "02_main_model_building.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root")

MODELS_DIR = PROJECT_ROOT / "MODELS"
DATA_CSV = PROJECT_ROOT / "data" / "budgetfood.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "main_model_results"

def load_module(module_filename: str, module_name: str):
    module_path = MODELS_DIR / module_filename
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

main_model_module = load_module("02_main_model_building.py", "main_model_building")
PROJECT_ROOT


PosixPath('/Users/chloe_wxy/Desktop/D300 real')

In [2]:
# Data check (no auto-install / no auto-download in notebook)
if not DATA_CSV.exists():
    raise FileNotFoundError(
        f"Missing {DATA_CSV}.\n"
        "Please put budgetfood.csv into data/ first, then rerun."
    )
DATA_CSV


PosixPath('/Users/chloe_wxy/Desktop/D300 real/data/budgetfood.csv')

## 1. Sample Used In The Main Model

The next outputs describe the estimation sample actually used in the main causal analysis. They combine a compact descriptive summary of the model variables with the overlap diagnostics used to justify later treatment-effect estimation.


In [3]:
import pandas as pd
from IPython.display import display

load_and_prepare = main_model_module.load_and_prepare
overlap_diagnostics = main_model_module.overlap_diagnostics

df, x_cols = load_and_prepare(csv_path=DATA_CSV, town_threshold=4)
overlap_summary, _ = overlap_diagnostics(df, x_cols)

sample_facts = pd.DataFrame([
    {"metric": "n_obs", "value": len(df)},
    {"metric": "treated_share", "value": df["D"].mean()},
    {"metric": "control_share", "value": 1 - df["D"].mean()},
])

main_sample_summary = (
    df[["Y", "D"] + x_cols]
    .agg(["mean", "std", "min", "median", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)

display(sample_facts)
display(main_sample_summary)
display(overlap_summary)


,metric,value
0,n_obs,23971.000000
1,treated_share,0.530641
2,control_share,0.469359


,variable,mean,std,min,median,max
0,Y,0.378320,0.165647,0.000000,0.364224,0.996560
1,D,0.530641,0.499071,0.000000,1.000000,1.000000
2,log_totexp,13.434573,0.722294,9.588845,13.502325,16.248909
3,size,3.693630,1.778939,1.000000,4.000000,17.000000
4,age,50.539819,15.106714,16.000000,50.000000,99.000000
5,sex_male,0.860373,0.346607,0.000000,1.000000,1.000000


,metric,value
0,n_obs,23971.000000
1,treated_n,12720.000000
2,control_n,11251.000000
3,treatment_rate,0.530641
4,propensity_min,0.059878
5,propensity_p01,0.196521
6,propensity_p99,0.820946
7,propensity_max,0.942980
8,propensity_overlap_lb_05,0.342981
9,propensity_overlap_ub_95,0.712780


## 2. Run The Main Estimation Pipeline

This cell estimates the parametric benchmarks, `LinearDML`, and `CausalForestDML`, then writes the saved result files to `outputs/main_model_results/`.


In [4]:
import pandas as pd
from contextlib import redirect_stdout
from io import StringIO
from IPython.display import Markdown, display

run_pipeline = main_model_module.run_pipeline
buffer = StringIO()
with redirect_stdout(buffer):
    run_pipeline(
        csv_path=DATA_CSV,
        output_dir=OUTPUT_DIR,
        seed=42,
        town_threshold=4,
    )

main_table = pd.read_csv(OUTPUT_DIR / "main_results.csv").rename(columns={
    "model": "Model",
    "ate_test": "ATE",
    "ate_ci_low": "CI low",
    "ate_ci_high": "CI high",
})
for col in ["ATE", "CI low", "CI high"]:
    main_table[col] = main_table[col].map(lambda x: f"{x:.4f}")

sample_text = "Sample used: 23,971 observations; treated share = 0.531."
display(Markdown(f"**Main results table**  \n{sample_text}"))
display(main_table)


**Main results table**  
Sample used: 23,971 observations; treated share = 0.531.

,Model,ATE,CI low,CI high
0,GWL_Baseline_HC3,-0.0263,-0.0304,-0.0222
1,GWL_Extended_HC3,-0.0260,-0.0302,-0.0218
2,LinearDML,-0.0241,-0.0282,-0.0200
3,CausalForestDML,-0.0244,-0.0454,-0.0034
